# Canada (CAD) — Fixed Income Derivatives

CORRA swaps, CGB sovereign bonds, and RRB (Real Return Bonds).

**Key features:**
- CORRA: Canadian Overnight Repo Rate Average (BOC)
- CGB: Canadian Government Bond (semi-annual, ACT/ACT)
- RRB: Real Return Bond (CPI-linked, deflation floor)

In [ ]:
import sys, os, math
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "python"))

import numpy as np
from datetime import date
from dateutil.relativedelta import relativedelta

from pricebook.fixed_income.canadian import (
    CORRASwap, RRBBond,
    build_corra_curve, synthetic_corra_strip,
)
from pricebook.fixed_income.sovereign_bonds import create_sovereign_bond
from pricebook.fixed_income.inflation_unit import dual_curve_breakeven
from pricebook.viz import configure_theme
from pricebook.viz._backend import apply_theme, create_figure

configure_theme(seaborn_style="whitegrid", seaborn_context="notebook")

REF = date(2024, 11, 4)
print(f"Reference date: {REF}")
print(f"CORRA (BOC policy): 4.25%")

## 1. CORRA Curve Construction

In [ ]:
strip = synthetic_corra_strip(REF, corra=0.0425)
corra_curve = build_corra_curve(REF, strip)

print(f"CORRA Swap Strip ({len(strip)} points):")
print(f"{'Tenor':>8}  {'Rate':>8}  {'DF':>10}")
print(f"{'─'*8}  {'─'*8}  {'─'*10}")
for c in strip:
    print(f"{c['months']:>6}M  {c['rate']*100:>7.2f}%  {corra_curve.df(c['maturity']):>10.6f}")

with apply_theme():
    fig, (ax1, ax2) = create_figure(2)
    tenors = [c["years"] for c in strip]
    ax1.plot(tenors, [c["rate"]*100 for c in strip], 'o-', linewidth=2)
    ax1.set_xlabel("Tenor (years)")
    ax1.set_ylabel("CORRA Rate (%)")
    ax1.set_title("CORRA Swap Curve — Nov 2024")
    ax1.axhline(4.25, color='gray', linestyle='--', alpha=0.5, label='Policy 4.25%')
    ax1.legend()

    ax2.plot(tenors, [corra_curve.df(c["maturity"]) for c in strip], 's-', linewidth=2, color='#059669')
    ax2.set_xlabel("Tenor (years)")
    ax2.set_ylabel("Discount Factor")
    ax2.set_title("CORRA Discount Factors")
    fig.tight_layout()

## 2. CORRA Swap Pricing

In [ ]:
print(f"CORRA Swap Pricing (CAD 100M notional):")
print(f"{'Tenor':>6}  {'Par Rate':>10}  {'DV01 (CAD)':>12}")
print(f"{'─'*6}  {'─'*10}  {'─'*12}")
for t in [1, 2, 3, 5, 7, 10, 20, 30]:
    swap = CORRASwap(REF, REF + relativedelta(years=t), 0.0425, notional=100e6)
    r = swap.price(corra_curve)
    print(f"{t:>4}Y  {r.par_rate*100:>9.2f}%  {r.dv01:>12,.0f}")

## 3. CGB — Canadian Government Bond

In [ ]:
print(f"CGB Pricing via CORRA Curve:")
print(f"{'Coupon':>8}  {'Tenor':>6}  {'Dirty Price':>12}")
print(f"{'─'*8}  {'─'*6}  {'─'*12}")
for cpn in [0.025, 0.035, 0.05]:
    for t in [2, 5, 10, 30]:
        cgb = create_sovereign_bond("CGB_CA", REF, REF + relativedelta(years=t), cpn)
        px = cgb.dirty_price(corra_curve)
        above = "↑" if px > 100 else "↓"
        print(f"{cpn*100:>7.2f}%  {t:>4}Y  {px:>11.4f} {above}")
    print()

## 4. RRB — Real Return Bond (CPI-Linked)

Canada's inflation-linked sovereign. Semi-annual real coupon. CPI ratio = current CPI / base CPI. Deflation floor: principal never falls below par.

In [ ]:
# Real curve at ~1.5% (Canadian real rates are low)
from pricebook.core.discount_curve import DiscountCurve
from pricebook.core.interpolation import InterpolationMethod

real_dates = [REF + relativedelta(years=y) for y in [1, 2, 5, 10, 20, 30]]
real_dfs = [math.exp(-0.015 * y) for y in [1, 2, 5, 10, 20, 30]]
real_curve = DiscountCurve(REF, real_dates, real_dfs, interpolation=InterpolationMethod.LOG_LINEAR)

base_cpi = 130.0
current_cpi = 160.0
cpi_ratio = max(current_cpi / base_cpi, 1.0)  # deflation floor

print(f"RRB Pricing (base CPI={base_cpi}, current CPI={current_cpi}, ratio={cpi_ratio:.4f}):")
print(f"{'Tenor':>6}  {'Real Cpn':>10}  {'Real PV':>10}  {'Nominal PV':>12}  {'CPI Ratio':>10}")
print(f"{'─'*6}  {'─'*10}  {'─'*10}  {'─'*12}  {'─'*10}")

for t, cpn in [(10, 0.015), (20, 0.02), (30, 0.015)]:
    rrb = RRBBond(REF, REF + relativedelta(years=t), cpn, base_cpi=base_cpi)
    r = rrb.price(REF, real_curve, current_cpi)
    print(f"{t:>4}Y  {cpn*100:>9.2f}%  {r.real_price:>10.4f}  {r.nominal_price:>12.4f}  {r.cpi_ratio:>10.4f}")

# BEI
bei = dual_curve_breakeven(corra_curve, real_curve, [2, 5, 10, 20, 30])
print(f"\nCanadian Breakeven Inflation:")
for row in bei:
    print(f"  {row['years']:>2}Y: {row['bei']*100:.2f}%")

with apply_theme():
    fig, axes = create_figure(1)
    ax = axes[0]
    years = [r["years"] for r in bei]
    ax.plot(years, [r["nominal_rate"]*100 for r in bei], 'o-', label='Nominal (CORRA)', linewidth=2)
    ax.plot(years, [r["real_rate"]*100 for r in bei], 's-', label='Real (CPI)', linewidth=2)
    ax.fill_between(years, [r["real_rate"]*100 for r in bei],
                     [r["nominal_rate"]*100 for r in bei], alpha=0.2, label='BEI')
    ax.set_xlabel("Tenor (years)")
    ax.set_ylabel("Rate (%)")
    ax.set_title("Canada — Breakeven Inflation Term Structure")
    ax.legend()
    fig.tight_layout()

## Summary

| Instrument | Convention | Key Feature |
|---|---|---|
| CORRA Swap | ACT/365, overnight | BOC reference rate |
| CGB | Semi-annual, ACT/ACT | G10 sovereign |
| RRB | Semi-annual, CPI-linked | Deflation floor on principal |
| BEI | CORRA - Real | ~2.8% implied inflation |